# Build a Tool-Calling Agentic AI Research Assistant with LangChain

Agents are systems that use an LLM as a reasoning engine to determine which actions to take and what the inputs to those actions should be. The results of those actions can then be fed back into the agent and it determines whether more actions are needed, or whether it is okay to stop.

![](https://i.imgur.com/1uVnBAm.png)

## Create Tools

Here we create two custom tools which are wrappers on top of the [Tavily API](https://tavily.com/#api) and [WeatherAPI](https://www.weatherapi.com/)

- Web Search tool with information extraction
- Weather tool

![](https://i.imgur.com/TyPAYXE.png)

In [1]:
import os
from tqdm import tqdm
from langchain.tools import tool
from markitdown import MarkItDown
from langchain_tavily import TavilySearch

import requests
from concurrent.futures import ThreadPoolExecutor, TimeoutError

tavily_tool = TavilySearch(max_results=5,
                            search_depth='advanced',
                            include_answers=False,
                            include_raw_content=True)

# certain websites won't let you crawl them unless you specify a user-agent
# pretending to be a real browser
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br"
})

md = MarkItDown(requests_session=session)

@tool
def search_web_extract_info(query:str) -> list:
    """Search the web for a query and extracts useful information from the search links"""
    print('Calling web search tool')
    results = tavily_tool.invoke(query)['results']
    docs = []
    
    def extract_content(url):
        """Helper function to extract content from a URL."""
        extracted_info = md.convert(url)
        text_title = extracted_info.title.strip()
        text_content = extracted_info.text_content.strip()
        return text_title + '\n' + text_content
    
    def fallback_doc(result):
        text_title = result['title'].strip()
        text_content = result['content'].strip()
        return f"{text_title}\n{text_content}"

    # parallelize execution of different urls
    with ThreadPoolExecutor() as executor:
        for result in tqdm(results):
            try:
                future = executor.submit(extract_content, result['url'])
                # Wait for up to 15 seconds for the task to complete
                content = future.result(timeout=15)
                docs.append(content)
            except TimeoutError:
                print(f"Extraction timed out for url: {result['url']}")
                docs.append(fallback_doc(result))
            except Exception as e:
                print(f"Error extracting from url: {result['url']} - {e}")
                docs.append(fallback_doc(result))

    return docs

@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather."""
    print('Calling weather tool')
    WEATHER_API_KEY=os.getenv("WEATHER_API_KEY")
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}"

    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

/media/ultron.legacy/HDD/GitHub/ai-agent-lab/.venv/lib/python3.10/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/media/ultron.legacy/HDD/GitHub/ai-agent-lab/.venv/lib/python3.10/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


## Test Tool Calling with LLM

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tools = [search_web_extract_info, get_weather]

llm_with_tools = llm.bind_tools(tools)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
prompt = "Get details of Microsoft's earnings call Q4 2024"
response = llm_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'search_web_extract_info',
  'args': {'query': 'Microsoft earnings call Q4 2024'},
  'id': '00c073ed-75e8-49cc-a43d-1e57932dd050',
  'type': 'tool_call'}]

In [4]:
prompt = "how is the weather in Bangalore today"
response = llm_with_tools.invoke(prompt)
response.tool_calls

[{'name': 'get_weather',
  'args': {'query': 'Bangalore'},
  'id': '30012614-7b17-4221-8ae4-c76d4a5e5e7c',
  'type': 'tool_call'}]

## Build and Test AI Agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using a tool calling agent to bind the tools to the agent with a prompt. We will also add in the capability to store historical conversations as memory

In [32]:
SYS_PROMPT = """Act as a helpful assistant.
                You run in a loop of Thought, Action, PAUSE, Observation.
                At the end of the loop, you output an Answer.
                Use Thought to describe your thoughts about the question you have been asked.
                Use Action to run one of the actions available to you - then return PAUSE.
                Observation will be the result of running those actions.
                Repeat till you get to the answer for the given user query.

                Use the following workflow format:
                  Question: the input task you must solve
                  Thought: you should always think about what to do
                  Action: the action to take which can be any of the following:
                            - break it into smaller steps if needed
                            - see if you can answer the given task with your trained knowledge
                            - call the most relevant tools at your disposal mentioned below in case you need more information
                  Action Input: the input to the action
                  Observation: the result of the action
                  ... (this Thought/Action/Action Input/Observation can repeat N times)
                  Thought: I now know the final answer
                  Final Answer: the final answer to the original input question

                Tools at your disposal to perform tasks as needed:
                  - get_weather: whenever user asks get the weather of a place.
                  - search_web_extract_info: whenever user asks for specific information or if you don't know the answer.
             """

Now, we can initalize the agent with the LLM, the prompt, and the tools.

The agent is responsible for taking in input and deciding what actions to take.
Note that we are passing in the model `llm`, not `llm_with_tools`.

That is because `create_agent` will call `.bind_tools` for us under the hood.
And also execute the `tool` suggested by `llm`.

This should ideally be used with an LLM which supports tool \ function calling

In [11]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
tools = [search_web_extract_info, get_weather]

agent = create_agent(
            model=llm, 
            tools =tools,
            system_prompt=SYS_PROMPT
        )

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Querying with llm, to check its knowledge.

In [15]:
query = """Summarize the key points discussed in Nvidia's Q1 2025 earnings call"""
response = llm.invoke(query)
print(response.content)

Nvidia's Q1 2025 earnings call highlighted another quarter of record-breaking financial performance, primarily driven by insatiable demand for its AI data center GPUs.

Here are the key points discussed:

1.  **Explosive Financial Performance:**
    *   **Record Revenue:** Achieved \$26.04 billion in revenue, up 18% sequentially and a staggering 262% year-over-year, significantly beating analyst expectations.
    *   **Strong Profitability:** Non-GAAP diluted EPS was \$6.12, up 19% sequentially and 461% year-over-year, also crushing estimates. Gross margin remained exceptionally high at 78.4% (non-GAAP).
    *   **Q2 Outlook:** Provided very strong guidance for Q2 2025, projecting revenue of \$28.0 billion (plus or minus 2%), indicating continued rapid growth.

2.  **Data Center Dominance (The Core Story):**
    *   **Record Data Center Revenue:** The Data Center segment was the primary driver, generating \$22.6 billion in revenue, up 23% sequentially and 427% year-over-year.
    *   *

Querying with Agent, to call tool inorder to get info about Nvidia's Q1 2025 earnings.

In [16]:
query = """Summarize the key points discussed in Nvidia's Q1 2025 earnings call"""
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

Calling web search tool


  0%|          | 0/5 [00:00<?, ?it/s]Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.
 20%|██        | 1/5 [00:01<00:07,  1.84s/it]

Error extracting from url: https://www.youtube.com/watch?v=CTx5aH3r67o - 'NoneType' object has no attribute 'strip'


 40%|████      | 2/5 [00:02<00:02,  1.07it/s]

Error extracting from url: https://en.macromicro.me/blog/nvidia-q1-2025-earnings-report-5-key-themes-driving-ai-s-next-phase - 403 Client Error: Forbidden for url: https://en.macromicro.me/blog/nvidia-q1-2025-earnings-report-5-key-themes-driving-ai-s-next-phase


...

Error extracting from url: https://www.investing.com/news/transcripts/earnings-call-transcript-nvidia-beats-q1-2025-expectations-stock-up-43-93CH-4069071 - 403 Client Error: Forbidden for url: https://www.investing.com/news/transcripts/earnings-call-transcript-nvidia-beats-q1-2025-expectations-stock-up-43-93CH-4069071


 80%|████████  | 4/5 [00:03<00:00,  1.66it/s]

Error extracting from url: https://www.gurufocus.com/stock/NVDA/transcripts/2444301 - 403 Client Error: Forbidden for url: https://www.gurufocus.com/stock/NVDA/transcripts/2444301


100%|██████████| 5/5 [00:05<00:00,  1.14s/it]


{'messages': [HumanMessage(content=["Summarize the key points discussed in Nvidia's Q1 2025 earnings call"], additional_kwargs={}, response_metadata={}, id='926da8d1-a2a9-4f00-a238-a7f937011fff'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web_extract_info', 'arguments': '{"query": "Nvidia Q1 2025 earnings call key points"}'}, '__gemini_function_call_thought_signatures__': {'a5bea1e3-11e9-4662-8c62-aa2a3a9ceb17': 'Co8CAXLI2nw9KFIUZr1Mk2j94HbWngTOX6GiuS+FFpIpR8zh/PuZQFI+m7L0Mp64TG/vipWnCSeoeGA6fMN6RjptR9IDnYBPNgh1rcivNGuhrEQcVNYjvX0Gy48xyUWNcGvAuShXz/0BQ9yZnz/dWTEo1suwQ42ySIyWUxkrEro4NxygWSGGcz+D76/WElmSg3nKTDQnQTu+9Ku+8GlZFqGSF1YxpyGJBpZOTMyu1EXv4zWbhRyQ1z4fphcn0HOLejVcV1KC+REv4AM1sgxt/hg4+SdcT2Q+cDMW6E/pNJadqW7h33heILl5Tkpyq1zC/1+qz75D00aDGtlpkTy2Jo33dBc4oGMI9/Tgu33ODZXCng=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b6e24-01e9-7f02-97dd-a205

In [27]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

print("Final Answer:")
display(Markdown(answer))

Final Answer:


Nvidia's Q1 2025 earnings call highlighted several key points:

*   **Record Financial Performance:** The company reported a record quarterly revenue of $26.0 billion, marking an 18% increase from the previous quarter and a substantial 262% rise year-over-year.
*   **Data Center as Primary Growth Driver:** Data Center revenue reached an all-time high of $22.6 billion, surging 23% quarter-over-quarter and 427% year-over-year. This exceptional growth is attributed to strong demand for generative AI training and inference on the Hopper platform, expanding to a wide range of customers beyond cloud service providers, including consumer internet, enterprise, sovereign AI, automotive, and healthcare sectors.
*   **AI Industrial Revolution:** CEO Jensen Huang emphasized that a "new industrial revolution" has commenced, with companies and nations actively partnering with Nvidia to transition to accelerated computing and establish "AI factories."
*   **New Product Platforms:** The Blackwell platform is now in full production, laying the groundwork for generative AI at trillion-parameter scale. Additionally, Spectrum-X was introduced, opening up a new market for large-scale AI in Ethernet-only data centers. Nvidia NIM, a new software offering, provides enterprise-grade, optimized generative AI across various environments.
*   **Shareholder Benefits:** Nvidia announced a ten-for-one forward stock split, effective June 7, 2024, to enhance stock accessibility. The quarterly cash dividend was also significantly increased by 150% to $0.01 per share on a post-split basis. The company returned a record $14.3 billion to shareholders in Q1.
*   **Gaming and Automotive Recovery:** Gaming revenue showed a strong recovery, growing 18% year-over-year to $2.6 billion, driven by the successful launch of the 50-series graphics cards and new AI gaming technologies. Automotive revenue also saw growth, reaching $329 million, with the next-generation NVIDIA DRIVE Thor platform being adopted by several electric vehicle manufacturers, and the introduction of the Project GR00T foundation model for humanoid robots.
*   **Professional Visualization:** This segment generated $427 million in revenue, up 45% year-over-year, with new RTX professional GPUs and Omniverse Cloud APIs being launched.
*   **Q2 Fiscal 2025 Outlook:** Nvidia anticipates revenue of $28.0 billion (plus or minus 2%) for the second quarter of fiscal 2025. Full-year gross margins are projected to be in the mid-70% range, with operating expenses expected to grow in the low-40% range.
*   **Export Control Impact:** The company acknowledged new U.S. government export controls on its H20 data center GPU, specifically designed for the China market.

Querying with Agent, to call tool inorder to get info about Intel's Q1 2025 earnings.

In [28]:
query = """Summarize the key points discussed in Intel's Q1 2025 earnings call"""
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

Calling web search tool


 20%|██        | 1/5 [00:00<00:01,  3.38it/s]

Error extracting from url: https://www.investing.com/news/transcripts/earnings-call-transcript-intel-beats-q1-2025-earnings-stock-declines-93CH-4003648 - 403 Client Error: Forbidden for url: https://www.investing.com/news/transcripts/earnings-call-transcript-intel-beats-q1-2025-earnings-stock-declines-93CH-4003648


100%|██████████| 5/5 [00:06<00:00,  1.37s/it]

Error extracting from url: https://download.intel.com/newsroom/2025/corporate/67s2p/Intel-1Q2025-Earnings.pdf - File conversion failed after 1 attempts:
 - PdfConverter threw MissingDependencyException with message: PdfConverter recognized the input as a potential .pdf file, but the dependencies needed to read .pdf files have not been installed. To resolve this error, include the optional dependency [pdf] or [all] when installing MarkItDown. For example:

* pip install markitdown[pdf]
* pip install markitdown[all]
* pip install markitdown[pdf, ...]
* etc.



{'messages': [HumanMessage(content=["Summarize the key points discussed in Intel's Q1 2025 earnings call"], additional_kwargs={}, response_metadata={}, id='a92a8d75-0a40-4160-92e5-c4ff25dbf53b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web_extract_info', 'arguments': '{"query": "Intel Q1 2025 earnings call summary"}'}, '__gemini_function_call_thought_signatures__': {'11897c6e-f05f-4a2b-84d0-100e3ae938cd': 'CrMCAXLI2nzLqs3W6AecLchZRj+Uyxy794Ama86uS4G7kl5u3hn0lJ4RIJikvsXJWSZkKCZFHcPNKVACA0xPTSX8zc/G0wEKmxC7wnPlzofIgEqps0cqwo7tcSYqzAYuFSpgDVLI6X8UvqOIULF+Ej/heqqpHqJH0CwLZ+Y75wbgu42zywfdAH+WEzxAsaqIGLM0caE4MafLucCobrGSJ88xvjx6Fz33jLS6WxSHLI4GtP2JE8jtxIEedbA4qu5tZ0wzu0Bwu+a4lhWOZSdSMB2mPlNZizj7XHYLAl+obRAXCncQqSrpxkCBN7zq/7hPErCh8veFfFA9SrqQ4AsjFCdOtJ/kEJHGUP1oO4A8Wjma8q5x9mCm1Fch5MPgXCA8UzAUywJaNeHDgAkfc7efaayyLA90hQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'

In [29]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

print("Final Answer:")
display(Markdown(answer))

Final Answer:


Intel's Q1 2025 earnings call highlighted a mixed performance, with the company beating its own guidance but issuing a weaker outlook for the second quarter. Key points include:

**Financial Performance (Q1 2025):**
*   **Revenue:** $12.7 billion, flat year-over-year, and at the high end of guidance.
*   **Non-GAAP Gross Margin:** 39.2%, exceeding guidance by 3 percentage points.
*   **Non-GAAP Earnings Per Share (EPS):** $0.13, surpassing the breakeven guidance.
*   **GAAP Net Loss:** $800 million, or $0.19 per share, compared to a $400 million loss in Q1 2024.
*   **Operating Cash Flow:** $800 million.
*   **Adjusted Free Cash Flow:** Negative $3.7 billion.

**Business Unit Performance (Q1 2025 vs. Q1 2024):**
*   **Intel Products Revenue:** Decreased by 3% to $11.8 billion.
    *   Client Computing Group (CCG) revenue fell 8% to $7.6 billion.
    *   Data Center and AI (DCAI) revenue grew 8% to $4.1 billion.
*   **Intel Foundry Revenue:** Increased by 7% to $4.7 billion, though it recorded an operating loss of $2.3 billion.

**Q2 2025 Guidance:**
*   **Revenue:** Projected to be between $11.2 billion and $12.4 billion, which is below analyst estimates.
*   **Gross Margin:** Expected to be approximately 36.5%.
*   **EPS:** Forecasted to be breakeven, also below analyst expectations.

**Strategic Initiatives and Challenges:**
*   **Cost Reduction:** Intel plans to significantly cut operational expenses to $17 billion in 2025 (from a previous target of $17.5 billion) and $16 billion in 2026. Capital expenditures are also being reduced to $18 billion in 2025 (from $20 billion), which will involve job cuts, particularly for management layers.
*   **New Leadership:** This was the first earnings call under new CEO Lip-Bu Tan, who emphasized simplifying operations, reducing bureaucracy, and strengthening engineering talent. Sachin Katti was named CTO and Head of AI.
*   **Macroeconomic Headwinds:** The company cited ongoing macroeconomic uncertainty, including fluid trade policies and persistent inflation, leading to a growing probability of an economic slowdown.
*   **Capacity Constraints:** Intel expects capacity constraints in its Intel 7 manufacturing process to continue.
*   **AI Focus:** Intel is prioritizing optimizing for emerging AI workloads, exploring new architectures for edge and inference applications, and fostering partnerships.
*   **Workplace Policy:** Employees will be required to work in the office four days a week starting in September.

Get the comparision between Nvidia VS Intel Q1 2025 earning.
But there is no history parameter, so our agent could recall the past conversations.

In [30]:
query = """which company's future outlook looks to be better?"""
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

{'messages': [HumanMessage(content=["which company's future outlook looks to be better?"], additional_kwargs={}, response_metadata={}, id='00981a5c-7cae-49fa-b077-7931d7aac914'),
  AIMessage(content=[{'type': 'text', 'text': 'Thought: The user is asking for a comparison of the future outlook of companies. To provide a meaningful answer, I need to know which companies the user wants to compare. Without specific companies, I cannot evaluate their future outlooks. I need to inform the user that I require more information.\nFinal Answer: Please specify the companies you would like me to compare for their future outlook.', 'extras': {'signature': 'CrENAXLI2nwl20OcOwTWjm6Jw9nGQq9OCvbenVro2P+So3p4g7OpFumDuf4fNKKXyAemNfFuSkEvlEMaCCL5BDLXhCG+DY+M+mghqphN9YWqLO61O5QhkWrXiK1edqGtzFYmizbzqS7KLneVlXvr0KoW8o5kMDjwWlNfXTE+POaKM37Ekt0Q99qmaxTDtnUd1zu5La2YSfVAcszpbTOaPtRKeqWRTnCCkpwQF3iL04vbUef1N58Bx1u4NiH1ynIYrNtSqI8bsse0P3ZrS3OjsFDeVmj1eWDKpBfFHtfiwrpSTUGqQR/5ASiGeY7n97Mp6PIIJ80zfHsKgg1009lhg0O3EM+kg

In [31]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

print("Final Answer:")
display(Markdown(answer))

Final Answer:


Thought: The user is asking for a comparison of the future outlook of companies. To provide a meaningful answer, I need to know which companies the user wants to compare. Without specific companies, I cannot evaluate their future outlooks. I need to inform the user that I require more information.
Final Answer: Please specify the companies you would like me to compare for their future outlook.

Querying with Agent, to call tool inorder to get info about Bangalore weather.

In [33]:
query = """how is the weather in Bangalore today?
           show detailed statistics
        """
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

Calling weather tool


{'messages': [HumanMessage(content=['how is the weather in Bangalore today?\n           show detailed statistics\n        '], additional_kwargs={}, response_metadata={}, id='03d58cc9-37d4-4e22-b6a9-20fc3a34ae6a'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"query": "Bangalore"}'}, '__gemini_function_call_thought_signatures__': {'e3b22e59-1640-4042-88e8-39c33e1c9c7c': 'CtsBAXLI2nwYf/at3YFWFnW/czMiNHtqDY1nlZwcNC76rC98kionOds8gh6xHWHJeyFQJMbrx8Kv795LBjjXD0DYpUWwu38OMGgBZEIuDSg56eNEyLe2xjYHr5F+nGVaYLc9iBkU/WElTdxTifL6WjIsXzhoznYZQBchaZ/N3XoYZAHHT8kbKyVIYwpZN1PvFhjWqvSLV6X/qAX1kM/V1daDSFecgrmQg3jlxsYmbvxS+SzHluQUKIGfD20GvVCjqU83MVmIhCYrK+DvxYzegaFQScdiqi+bLXgHhsGo'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b6e3a-5918-75e2-b201-3cdd9e5f9ef4-0', tool_calls=[{'name': 'get_weather', 'args': {'query': 'Bangalore'}, 'id': 'e3b22e59-

In [34]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

display(Markdown(answer))

The weather in Bangalore today, December 30, 2025, at 13:15, is as follows:

**Condition:** Mist
**Temperature:** 26.3°C (79.3°F)
**Feels like:** 26°C (78.7°F)
**Humidity:** 54%
**Cloud cover:** 50%
**Wind:** 5.4 kph (3.4 mph) from the East (83 degrees)
**Wind gust:** 6.2 kph (3.9 mph)
**Pressure:** 1017 mb (30.03 in)
**Precipitation:** 0 mm (0 in)
**Visibility:** 5 km (3 miles)
**UV Index:** 8.4
**Dewpoint:** 10.5°C (50.9°F)
**Heat Index:** 26.7°C (80.1°F)
**Windchill:** 27.3°C (81.1°F)

Querying with Agent, to call tool inorder to get info about Dubai weather.

In [35]:
query = """how is the weather in Dubai today?
           show detailed statistics
        """
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

Calling weather tool


{'messages': [HumanMessage(content=['how is the weather in Dubai today?\n           show detailed statistics\n        '], additional_kwargs={}, response_metadata={}, id='ab6bc905-0275-4ba7-b4a5-6f9acfe7dcd2'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"query": "Dubai"}'}, '__gemini_function_call_thought_signatures__': {'c02d8b46-23ab-407b-8ee9-0b432dd0c190': 'CvwBAXLI2nyXjHc4w3i9gUbNfH2McA0RMmZfyx0ZlERK3WlyklZcEeNsraZkCr8QgTJ293yuiS/fqXx1r/hymQ61jO8E7J1Q5g21vbcwvcl5ibmkMtRbtj6Z5JEVBB4Ozq6ov5Ta4fnoMoKDsGQPFLz6gdWuyqN3JE7+Xz7H7NLTXnrBWa5juUR0fqdpEqfY9M/vIgXRAIjN5D2pHKcCRXIKsl3oXBUHy3oueJz94NrJy4G8+L8/+AoiQhNDnlaze314p46rfQtPPwb9howdZ2Z5U2fieAI7mdDqMYXKrWANBXTzwcOTAQnZ026QKLrtSym1+h36SlWhvxfeDzwh'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b6e3a-82cd-7f51-b2e4-1991419d43f1-0', tool_calls=[{'name': 'get_weather', 'args': {'qu

In [36]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

display(Markdown(answer))

The weather in Dubai today is sunny with a temperature of 25.1°C (77.2°F). It feels like 25.9°C (78.5°F). The wind is from the WNW at 36.4 kph (22.6 mph), with gusts up to 46 kph (28.6 mph). Humidity is at 30%, and there is 25% cloud cover. The pressure is 1016 mb (30 in), and visibility is 10 km (6 miles). The UV index is 4.3. The dew point is 10.5°C (51°F), and the heat index is 24.3°C (75.8°F). The wind chill is 21.7°C (71.1°F). The last update was on 2025-12-30 at 11:45 AM local time.

Get the comparision between Bangalore and Dubai weather.
But there is no history parameter, so our agent could recall the past conversations.

In [37]:
query = """which city is hotter?
        """
response = agent.invoke({"messages": [{"role": "user", "content": {query}}]})
response

{'messages': [HumanMessage(content=['which city is hotter?\n        '], additional_kwargs={}, response_metadata={}, id='0171ad5b-5207-4485-b78d-ab806d6b10d5'),
  AIMessage(content=[{'type': 'text', 'text': 'Final Answer: Please tell me which cities you would like to compare.', 'extras': {'signature': 'CvcBAXLI2nwOfPWo1P0zrWKCx/4xWCFcPzuqWQ2d5sEGFQFAUUvAG1w3BoQSYf3M7vSzlIm6YIuoXlEk/PdeOkSD3oWeCbDTIJPDWZNO4WvY2fAZRFXYWzmUqA/7PCfK2+cwDcan1EVuGCda5HHBsBXLdGxIklMzfsWWHpxFqYl2TaGZd5PfowAK1xDEDHEc06dgBwhYt3AKC2X4FGjQhmolt4DcOAgMbE35DfaIfLbjfZPbvPPoNSohz+5NHm6rDteWyaDMM2ARzp2CeJmlW9+OH8/FR2ajsWMR5/FyX7r4eBb5ffKmwIfZi1TbnJ0+L7Bz4OTUaKCGVg=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b6e3b-422a-73a3-9420-6fa4f35fac3a-0', usage_metadata={'input_tokens': 406, 'output_tokens': 59, 'total_tokens': 465, 'input_token_details': {'cache_read': 0}, 'output_token_details': 

In [38]:
from IPython.display import display, Markdown

# Get the final AI response (last message)
final_message = response["messages"][-1]

# Or more robustly (handles both string and list content):
if isinstance(final_message.content, list):
    # Gemini returns list - get first text item
    answer = next((item["text"] for item in final_message.content if item["type"] == "text"), "")
else:
    answer = final_message.content

display(Markdown(answer))

Final Answer: Please tell me which cities you would like to compare.

The agent is doing pretty well but unfortunately it doesn't remember conversations. We will use some user-session based memory to store this and dive deeper into this in the next notebook.

- Mutli-user
- User based Memory